# Extract JSON-LD from Common Crawl

This notebook extracts JSON-LD semantic annotations from Common Crawl for a specific domain.
It queries the **Parquet columnar index** directly from S3, then fetches individual WARC records via HTTP range requests.

In [1]:
!pip install duckdb requests beautifulsoup4 warcio

Defaulting to user installation because normal site-packages is not writeable


In [2]:
!pip install --upgrade ipywidgets widgetsnbextension

Defaulting to user installation because normal site-packages is not writeable


In [1]:
import io
import json
import time
import pandas as pd

import duckdb
import requests
from bs4 import BeautifulSoup
from warcio.archiveiterator import ArchiveIterator

CC_DATA_BASE = "https://data.commoncrawl.org"

## Configuration

- `DOMAIN`: the registered domain (e.g. `lemonde.fr`, `reuters.com`)
- `URL_PREFIX`: optional path prefix to filter by (e.g. `https://www.lemonde.fr/en/`). Set to `None` to get all pages from the domain.
- `CRAWL_ID`: the Common Crawl crawl identifier. Check https://commoncrawl.org/overview for available crawls.

In [2]:
# DOMAIN = "lemonde.fr"                          # registered domain
# URL_PREFIX = "https://www.lemonde.fr/en/"       # set to None to skip path filtering
# LIMIT = 100                                     # max pages to fetch
CRAWL_ID = "CC-MAIN-2026-12"                    # latest crawl as of April 2026
OUTPUT_FILE = "jsonld_results.jsonl"

## Step 1: Check which paths file is available

Common Crawl publishes two index listings per crawl:
- `cc-index.paths.gz` → CDX text files (NOT Parquet)
- `cc-index-table.paths.gz` → Parquet files (what we need)

If neither is available, we fall back to querying the global Parquet table on S3 directly.

In [3]:
for name in ["cc-index.paths.gz", "cc-index-table.paths.gz"]:
    url = f"{CC_DATA_BASE}/crawl-data/{CRAWL_ID}/{name}"
    resp = requests.head(url, timeout=30)
    print(f"{name}: HTTP {resp.status_code}")

cc-index.paths.gz: HTTP 200
cc-index-table.paths.gz: HTTP 200


## Step 2: Query the Parquet columnar index

We query the Parquet table directly from S3 using the partition path:
`s3://commoncrawl/cc-index/table/cc-main/warc/crawl={CRAWL_ID}/subset=warc/*.parquet`

This avoids needing any paths listing file. DuckDB reads the Parquet files via S3 (no AWS credentials needed — it's public data, but must use `us-east-1` region).

**Note:** This query can take several minutes as it scans large Parquet files remotely.

In [4]:
import gzip

paths_url = f"{CC_DATA_BASE}/crawl-data/{CRAWL_ID}/cc-index-table.paths.gz"
resp = requests.get(paths_url, timeout=60)
resp.raise_for_status()

paths = gzip.decompress(resp.content).decode().splitlines()

parquet_files = [
    f"https://data.commoncrawl.org/{p}"
    for p in paths
    if p.endswith(".parquet")
]

print(f"Found {len(parquet_files)} parquet files")

Found 900 parquet files


In [ ]:
import duckdb

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
# con.execute("PRAGMA enable_progress_bar;")
con.execute("PRAGMA disable_progress_bar;") 
records = []

# DOMAIN = "wikipedia.org"
# LIMIT = 50

DOMAINS = ["en.wikipedia.org"]
domains_sql = ", ".join(f"'{d}'" for d in DOMAINS)


for i, file in enumerate(parquet_files):
    try:
        if (i+1) % 50 == 0:
            print(f"[{i+1}/{len(parquet_files)}] Processing: {file}")

        rows = con.execute(f"""
            SELECT
                url,
                warc_filename,
                warc_record_offset,
                warc_record_length,
                url_host_name,
                url_host_registered_domain,
                fetch_time,
                content_languages,
                content_mime_detected,
                fetch_status
            FROM read_parquet('{file}')
            WHERE url_host_name IN ({domains_sql})
            AND fetch_status = 200
            AND content_mime_detected = 'text/html'
        """).fetchall()

        # rows = con.execute(f"""
        #     SELECT url, warc_filename, warc_record_offset, warc_record_length
        #     FROM read_parquet('{file}')
        #     WHERE url_host_registered_domain = '{DOMAIN}'
        #     LIMIT {LIMIT}
        # """).fetchall()

        for r in rows:
            records.append({
                "url": r[0],
                "warc_filename": r[1],
                "warc_record_offset": r[2],
                "warc_record_length": r[3],
                "url_host_name": r[4],
                "url_host_registered_domain": r[5],
                "fetch_time": r[6],
                "content_languages": r[7],
                "content_mime_detected": r[8],
                "fetch_status": r[9]
            })

        # if len(records) >= LIMIT:
        #     print("Reached LIMIT, stopping early.")
        #     break

    except Exception as e:
        print(f"Skipping broken file: {file}")
        continue

con.close()

print(f"Total records found: {len(records)}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[50/900] Processing: https://data.commoncrawl.org/cc-index/table/cc-main/warc/crawl=CC-MAIN-2026-12/subset=crawldiagnostics/part-00049-34e6fda8-4578-4144-973f-2adf16729186.c000.gz.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[100/900] Processing: https://data.commoncrawl.org/cc-index/table/cc-main/warc/crawl=CC-MAIN-2026-12/subset=crawldiagnostics/part-00099-34e6fda8-4578-4144-973f-2adf16729186.c000.gz.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (2 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Skipping broken file: https://data.commoncrawl.org/cc-index/table/cc-main/warc/crawl=CC-MAIN-2026-12/subset=crawldiagnostics/part-00196-34e6fda8-4578-4144-973f-2adf16729186.c000.gz.parquet
[200/900] Processing: https://data.commoncrawl.org/cc-index/table/cc-main/warc/crawl=CC-MAIN-2026-12/subset=crawldiagnostics/part-00199-34e6fda8-4578-4144-973f-2adf16729186.c000.gz.parquet


In [ ]:
# Save output to csv 
import pandas as pd

df = pd.DataFrame(records)
df.to_csv("records_wiki.csv", index=False)

### Quick look at what we found

In [11]:
records[:5]

[{'url': 'https://en.wikipedia.org/?curid=13729',
  'warc_filename': 'crawl-data/CC-MAIN-2026-12/segments/1772687277647.32/warc/CC-MAIN-20260308165213-20260308195213-00802.warc.gz',
  'warc_record_offset': 156364261,
  'warc_record_length': 139583,
  'url_host_name': 'en.wikipedia.org',
  'url_host_registered_domain': 'wikipedia.org',
  'fetch_time': datetime.datetime(2026, 3, 8, 18, 31, 26, tzinfo=<DstTzInfo 'Europe/Paris' CET+1:00:00 STD>),
  'content_languages': 'eng,dan',
  'content_mime_detected': 'text/html',
  'fetch_status': 200},
 {'url': 'https://en.wikipedia.org/?curid=1504834',
  'warc_filename': 'crawl-data/CC-MAIN-2026-12/segments/1772687553247.70/warc/CC-MAIN-20260316014608-20260316044608-00215.warc.gz',
  'warc_record_offset': 172446922,
  'warc_record_length': 28644,
  'url_host_name': 'en.wikipedia.org',
  'url_host_registered_domain': 'wikipedia.org',
  'fetch_time': datetime.datetime(2026, 3, 16, 5, 5, 44, tzinfo=<DstTzInfo 'Europe/Paris' CET+1:00:00 STD>),
  'conte

In [13]:
len(records)

102422

In [12]:
unique_domains = set(r["url_host_registered_domain"] for r in records)
print(unique_domains)

{'wikipedia.org'}


In [ ]:
# import duckdb

# con = duckdb.connect()
# con.execute("INSTALL httpfs; LOAD httpfs;")
# # con.execute("PRAGMA enable_progress_bar;")
# con.execute("PRAGMA disable_progress_bar;") 

# records = []

# # DOMAIN = "wikipedia.org"
# # LIMIT = 100

# DOMAINS = [
#     # Dedicated fact-checkers
#     "politifact.com",
#     "snopes.com",
#     "factcheck.org",
#     "fullfact.org",
#     "africacheck.org",
#     # News wire services
#     "reuters.com",
#     "apnews.com",
#     # International broadcasters
#     "bbc.com",
#     "bbc.co.uk",
#     "aljazeera.com",
#     "france24.com",
#     "dw.com",
#     # Major newspapers
#     "theguardian.com",
#     "nytimes.com",
#     "thetimes.com",
#     "lemonde.fr",
#     "cnn.com",
# ]
# # DOMAINS = ["bbc.com", "bbc.co.uk"]

# domains_sql = ", ".join(f"'{d}'" for d in DOMAINS)


# for i, file in enumerate(parquet_files):
#     try:
#         if (i+1) % 50 == 0:
#             print(f"[{i+1}/{len(parquet_files)}] Processing: {file}")

#         rows = con.execute(f"""
#             SELECT
#                 url,
#                 warc_filename,
#                 warc_record_offset,
#                 warc_record_length,
#                 url_host_name,
#                 url_host_registered_domain,
#                 fetch_time,
#                 content_languages,
#                 content_mime_detected,
#                 fetch_status
#             FROM read_parquet('{file}')
#             WHERE url_host_registered_domain IN ({domains_sql})
#         """).fetchall()

#         # rows = con.execute(f"""
#         #     SELECT url, warc_filename, warc_record_offset, warc_record_length
#         #     FROM read_parquet('{file}')
#         #     WHERE url_host_registered_domain = '{DOMAIN}'
#         #     LIMIT {LIMIT}
#         # """).fetchall()

#         for r in rows:
#             records.append({
#                 "url": r[0],
#                 "warc_filename": r[1],
#                 "warc_record_offset": r[2],
#                 "warc_record_length": r[3],
#                 "url_host_name": r[4],
#                 "url_host_registered_domain": r[5],
#                 "fetch_time": r[6],
#                 "content_languages": r[7],
#                 "content_mime_detected": r[8],
#                 "fetch_status": r[9]
#             })

#         # if len(records) >= LIMIT:
#         #     print("Reached LIMIT, stopping early.")
#         #     break

#     except Exception as e:
#         print(f"Skipping broken file: {file}")
#         continue

# con.close()

# print(f"Total records found: {len(records)}")

Skipping broken file: https://data.commoncrawl.org/cc-index/table/cc-main/warc/crawl=CC-MAIN-2026-12/subset=crawldiagnostics/part-00015-34e6fda8-4578-4144-973f-2adf16729186.c000.gz.parquet
[50/900] Processing: https://data.commoncrawl.org/cc-index/table/cc-main/warc/crawl=CC-MAIN-2026-12/subset=crawldiagnostics/part-00049-34e6fda8-4578-4144-973f-2adf16729186.c000.gz.parquet
Skipping broken file: https://data.commoncrawl.org/cc-index/table/cc-main/warc/crawl=CC-MAIN-2026-12/subset=crawldiagnostics/part-00071-34e6fda8-4578-4144-973f-2adf16729186.c000.gz.parquet


In [5]:
import duckdb

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("PRAGMA disable_progress_bar;")

records = []

DOMAINS = [
    # Dedicated fact-checkers
    "politifact.com",
    "snopes.com",
    "factcheck.org",
    "fullfact.org",
    "africacheck.org",
    # News wire services
    "reuters.com",
    "apnews.com",
    # International broadcasters
    "bbc.com",
    "bbc.co.uk",
    "aljazeera.com",
    "france24.com",
    "dw.com",
    # Major newspapers
    "theguardian.com",
    "nytimes.com",
    "thetimes.com",
    "lemonde.fr",
    "cnn.com",
]

domains_sql = ", ".join(f"'{d}'" for d in DOMAINS)

EXPECTED_COLS = [
    "url", "warc_filename", "warc_record_offset", "warc_record_length",
    "url_host_name", "url_host_registered_domain", "fetch_time",
    "content_languages", "content_mime_detected", "fetch_status"
]

for i, file in enumerate(parquet_files):
    try:
        if (i+1) % 50 == 0:
            print(f"[{i+1}/{len(parquet_files)}] Processing: {file}")

        available = set(
            con.execute(f"DESCRIBE SELECT * FROM read_parquet('{file}') LIMIT 0")
               .fetchdf()["column_name"]
               .str.lower()
        )

        select_clause = ", ".join(
            col if col in available else f"NULL AS {col}"
            for col in EXPECTED_COLS
        )

        where_clause = (
            f"WHERE url_host_registered_domain IN ({domains_sql})"
            if "url_host_registered_domain" in available else ""
        )

        rows = con.execute(f"""
            SELECT {select_clause}
            FROM read_parquet('{file}')
            {where_clause}
        """).fetchall()

        for r in rows:
            records.append({col: r[j] for j, col in enumerate(EXPECTED_COLS)})

    except Exception as e:
        print(f"Skipping broken file: {file}\n  Reason: {e}")
        continue

con.close()

print(f"Total records found: {len(records)}")


[50/900] Processing: https://data.commoncrawl.org/cc-index/table/cc-main/warc/crawl=CC-MAIN-2026-12/subset=crawldiagnostics/part-00049-34e6fda8-4578-4144-973f-2adf16729186.c000.gz.parquet
[100/900] Processing: https://data.commoncrawl.org/cc-index/table/cc-main/warc/crawl=CC-MAIN-2026-12/subset=crawldiagnostics/part-00099-34e6fda8-4578-4144-973f-2adf16729186.c000.gz.parquet
[150/900] Processing: https://data.commoncrawl.org/cc-index/table/cc-main/warc/crawl=CC-MAIN-2026-12/subset=crawldiagnostics/part-00149-34e6fda8-4578-4144-973f-2adf16729186.c000.gz.parquet
[200/900] Processing: https://data.commoncrawl.org/cc-index/table/cc-main/warc/crawl=CC-MAIN-2026-12/subset=crawldiagnostics/part-00199-34e6fda8-4578-4144-973f-2adf16729186.c000.gz.parquet
[250/900] Processing: https://data.commoncrawl.org/cc-index/table/cc-main/warc/crawl=CC-MAIN-2026-12/subset=crawldiagnostics/part-00249-34e6fda8-4578-4144-973f-2adf16729186.c000.gz.parquet
[300/900] Processing: https://data.commoncrawl.org/cc-in

In [6]:
# Save output to csv 
import pandas as pd

df = pd.DataFrame(records)
df.to_csv("records_other2.csv", index=False)

In [7]:
len(records)

133627

## Analysis of Domains

In [8]:

# ── 1. Shape & basic info ──────────────────────────────────────────
print(f"Total records: {len(df)}")
print(f"Columns: {list(df.columns)}\n")

# ── 2. Records per domain ──────────────────────────────────────────
print("=== Records per domain ===")
print(df["url_host_registered_domain"].value_counts().to_string(), "\n")

# ── 3. Fetch status per domain ─────────────────────────────────────
print("=== Fetch status per domain ===")
print(df.groupby(["url_host_registered_domain", "fetch_status"]).size().to_string(), "\n")

# ── 4. WARC subset breakdown (warc / crawldiagnostics / robotstxt) ─
df["warc_subset"] = df["warc_filename"].str.extract(r"subset=(\w+)/")
print("=== WARC subset per domain ===")
print(df.groupby(["url_host_registered_domain", "warc_subset"]).size().to_string(), "\n")

# ── 5. MIME type distribution ──────────────────────────────────────
print("=== MIME types ===")
print(df["content_mime_detected"].value_counts().to_string(), "\n")

# ── 6. Language distribution (non-null only) ───────────────────────
print("=== Content languages (top 10) ===")
print(df["content_languages"].dropna().value_counts().head(10).to_string(), "\n")

# ── 7. WARC record size stats ──────────────────────────────────────
print("=== WARC record size (bytes) ===")
print(df["warc_record_length"].describe().to_string(), "\n")

# ── 8. Fetch time range ────────────────────────────────────────────
print("=== Fetch time range ===")
print(f"  Earliest: {df['fetch_time'].min()}")
print(f"  Latest:   {df['fetch_time'].max()}")


Total records: 133627
Columns: ['url', 'warc_filename', 'warc_record_offset', 'warc_record_length', 'url_host_name', 'url_host_registered_domain', 'fetch_time', 'content_languages', 'content_mime_detected', 'fetch_status']

=== Records per domain ===
url_host_registered_domain
aljazeera.com      73704
thetimes.com       29625
politifact.com     13131
cnn.com             8184
factcheck.org       3493
fullfact.org        1551
theguardian.com     1456
dw.com              1075
apnews.com           949
france24.com         301
snopes.com           140
africacheck.org        9
reuters.com            3
bbc.com                3
nytimes.com            1
lemonde.fr             1
bbc.co.uk              1 

=== Fetch status per domain ===
url_host_registered_domain  fetch_status
africacheck.org             301                 1
                            403                 8
aljazeera.com               200             62693
                            301             10648
                      

In [9]:
df[df["url_host_registered_domain"] == "bbc.com"]

,url,warc_filename,warc_record_offset,warc_record_length,url_host_name,url_host_registered_domain,fetch_time,content_languages,content_mime_detected,fetch_status,warc_subset
31938,https://bbc.com/robots.txt,crawl-data/CC-MAIN-2026-12/segments/1772687277...,507487,908,bbc.com,bbc.com,2026-03-07 16:34:49+01:00,None,text/plain,301,NaN
31939,https://www.bbc.com/robots.txt,crawl-data/CC-MAIN-2026-12/segments/1772687277...,1266543,2513,www.bbc.com,bbc.com,2026-03-07 16:34:49+01:00,None,text/plain,200,NaN
31940,https://player.bbc.com/robots.txt,crawl-data/CC-MAIN-2026-12/segments/1772687277...,1154162,955,player.bbc.com,bbc.com,2026-03-10 13:08:35+01:00,None,text/x-robots,200,NaN


In [10]:
df_200 = df[df["fetch_status"] == 200]
df_200.to_csv("records_other_200.csv", index=False)
print(f"Records with 200: {len(df_200)}")


Records with 200: 101751


In [11]:
df_200_html = df[(df["fetch_status"] == 200) & (df["content_mime_detected"] == "text/html")]
df_200_html.to_csv("records_200_html.csv", index=False)
print(f"Records with 200 + text/html: {len(df_200_html)}")

Records with 200 + text/html: 100226


In [12]:
import csv
import requests
import time

ROBOTS_CSV = "records_other_200.csv"

# ── 1. Filter for robots.txt rows, then pick one URL per registered domain ─────
domain_url = {}
with open(ROBOTS_CSV) as f:
    for row in csv.DictReader(f):
        mime = row.get("content_mime_detected", "").strip()
        url  = row.get("url", "").strip()
        # Keep only rows that are actually robots.txt files
        if mime != "text/x-robots" and not url.endswith("/robots.txt"):
            continue
        domain = row["url_host_registered_domain"].strip()
        if domain and domain not in domain_url:
            domain_url[domain] = url

print(f"Unique domains with a robots.txt URL: {len(domain_url)}")
for d, u in domain_url.items():
    print(f"  {d:30s}  {u}")


# ── 2. Fetch each robots.txt and check for CCBot blocking ─────────────────────
def is_ccbot_blocked(text: str) -> bool:
    """
    Returns True if robots.txt blocks CCBot, defined as:
      - An explicit 'User-agent: CCBot' block with any non-empty Disallow, OR
      - A wildcard 'User-agent: *' block with 'Disallow: /' (blocks everything)
    """
    current_agents = set()

    for raw_line in text.splitlines():
        line = raw_line.split("#")[0].strip()   # strip inline comments
        if not line:
            current_agents = set()              # blank line ends the block
            continue

        if line.lower().startswith("user-agent:"):
            agent = line.split(":", 1)[1].strip().lower()
            current_agents.add(agent)
        elif line.lower().startswith("disallow:"):
            path = line.split(":", 1)[1].strip()
            if not path:
                continue
            # Explicit CCBot block with any disallowed path
            if "ccbot" in current_agents:
                return True
            # Wildcard block that disallows everything
            if "*" in current_agents and path == "/":
                return True

    return False


print("\n" + "=" * 60)
print("CCBot blocking check")
print("=" * 60)

blocked = []
allowed = []
errors  = []

for domain, url in domain_url.items():
    try:
        resp = requests.get(url, timeout=15, headers={"User-Agent": "research-bot/1.0"})
        resp.raise_for_status()
        blocked_flag = is_ccbot_blocked(resp.text)
        if blocked_flag:
            blocked.append(domain)
            status = "BLOCKED"
        else:
            allowed.append(domain)
            status = "allowed"
        print(f"  {domain:30s}  {status}  (HTTP {resp.status_code})")
    except Exception as e:
        errors.append((domain, str(e)))
        print(f"  {domain:30s}  ERROR: {e}")
    time.sleep(0.5)

print("\n" + "=" * 60)
print(f"Domains that BLOCK CCBot ({len(blocked)}):")
for d in blocked:
    print(f"  {d}")

if allowed:
    print(f"\nDomains that ALLOW CCBot ({len(allowed)}):")
    for d in allowed:
        print(f"  {d}")

if errors:
    print(f"\nErrors ({len(errors)}):")
    for d, e in errors:
        print(f"  {d}: {e}")

Unique domains with a robots.txt URL: 15
  nytimes.com                     https://cn.nytimes.com/robots.txt
  thetimes.com                    https://appointments.thetimes.com/robots.txt
  theguardian.com                 https://www.theguardian.com/robots.txt
  aljazeera.com                   https://www.aljazeera.com/robots.txt
  snopes.com                      https://www.snopes.com/robots.txt
  politifact.com                  https://www.politifact.com/robots.txt
  apnews.com                      https://apnews.com/robots.txt
  reuters.com                     https://jp.reuters.com/robots.txt
  bbc.com                         https://www.bbc.com/robots.txt
  cnn.com                         http://www.cnn.com/robots.txt
  factcheck.org                   https://www.factcheck.org/robots.txt
  bbc.co.uk                       https://www.bbc.co.uk/robots.txt
  france24.com                    https://www.france24.com/robots.txt
  fullfact.org                    https://fullfact.org/robo

In [13]:
import csv
from collections import defaultdict, Counter

CSV_FILE = "records_200_html.csv"

domain_hosts = defaultdict(list)

with open(CSV_FILE) as f:
    for row in csv.DictReader(f):
        domain = row["url_host_registered_domain"].strip()
        host   = row["url_host_name"].strip()
        domain_hosts[domain].append(host)

print(f"{'='*65}")
print(f"Subdomains per registered domain (records_200_html.csv)")
print(f"{'='*65}\n")

for domain, hosts in sorted(domain_hosts.items(), key=lambda x: -len(x[1])):
    host_counts = Counter(hosts)
    print(f"{domain}  ({len(hosts)} URLs total)")
    print(f"  {'subdomain':<40} {'count':>7}  {'%':>6}")
    print(f"  {'-'*55}")
    for host, count in host_counts.most_common():
        pct = count / len(hosts) * 100
        print(f"  {host:<40} {count:>7}  {pct:>5.1f}%")
    print()


Subdomains per registered domain (records_200_html.csv)

aljazeera.com  (62579 URLs total)
  subdomain                                  count       %
  -------------------------------------------------------
  www.aljazeera.com                          61888   98.9%
  liberties.aljazeera.com                      592    0.9%
  interactive.aljazeera.com                     62    0.1%
  remix.aljazeera.com                           19    0.0%
  inspiringstories.aljazeera.com                12    0.0%
  fmls.aljazeera.com                             3    0.0%
  mediaview.aljazeera.com                        2    0.0%
  lifeonhold.aljazeera.com                       1    0.0%

thetimes.com  (16735 URLs total)
  subdomain                                  count       %
  -------------------------------------------------------
  appointments.thetimes.com                  12871   76.9%
  www.thetimes.com                            3848   23.0%
  globalstore.thetimes.com                      13 

In [14]:
# Reads records_200_html.csv, groups URLs by registered domain and subdomain,
# classifies each URL by its top-level path pattern, and exports an Excel file
# (url_analysis.xlsx) with one sheet per domain showing the top 5 URL types
# per subdomain with counts, percentages, and 3 example URLs each.

import csv
from collections import defaultdict, Counter
from urllib.parse import urlparse
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment

# pip install openpyxl

CSV_FILE = "records_200_html.csv"
OUTPUT_FILE = "url_analysis.xlsx"

domain_host_urls = defaultdict(lambda: defaultdict(list))

with open(CSV_FILE) as f:
    for row in csv.DictReader(f):
        domain = row["url_host_registered_domain"].strip()
        host   = row["url_host_name"].strip()
        url    = row["url"].strip()
        domain_host_urls[domain][host].append(url)

def classify_url(url: str) -> str:
    path = urlparse(url).path.rstrip("/")
    if not path:
        return "homepage"
    parts = [p for p in path.split("/") if p]
    section = parts[0] if parts else ""
    return f"/{section}/..." if len(parts) > 1 else f"/{section}"

# Styles
HEADER_FONT    = Font(bold=True, color="FFFFFF")
HEADER_FILL    = PatternFill("solid", fgColor="2F4F7F")
DOMAIN_FONT    = Font(bold=True, color="FFFFFF")
DOMAIN_FILL    = PatternFill("solid", fgColor="4A7FB5")
SUBDOMAIN_FONT = Font(bold=True)
SUBDOMAIN_FILL = PatternFill("solid", fgColor="D9E8F5")
CENTER         = Alignment(horizontal="center")

COLUMNS = ["subdomain", "subdomain_urls", "subdomain_%",
           "url_pattern", "pattern_count", "pattern_%",
           "example_1", "example_2", "example_3"]
COL_WIDTHS = [35, 14, 12, 22, 14, 12, 60, 60, 60]

wb = openpyxl.Workbook()
wb.remove(wb.active)  # remove default sheet

for domain, host_urls in sorted(domain_host_urls.items(), key=lambda x: -sum(len(v) for v in x[1].values())):
    total_domain = sum(len(u) for u in host_urls.values())
    ws = wb.create_sheet(title=domain[:31])  # sheet names max 31 chars

    # Header row
    ws.append(COLUMNS)
    for col, width in enumerate(COL_WIDTHS, start=1):
        cell = ws.cell(row=1, column=col)
        cell.font      = HEADER_FONT
        cell.fill      = HEADER_FILL
        cell.alignment = CENTER
        ws.column_dimensions[cell.column_letter].width = width

    for host, urls in sorted(host_urls.items(), key=lambda x: -len(x[1])):
        pct_host = len(urls) / total_domain * 100

        pattern_urls = defaultdict(list)
        for url in urls:
            pattern_urls[classify_url(url)].append(url)

        type_counts = Counter({p: len(u) for p, u in pattern_urls.items()})
        top_patterns = type_counts.most_common(5)

        for i, (pattern, count) in enumerate(top_patterns):
            examples = pattern_urls[pattern][:3]
            row_data = [
                host if i == 0 else "",          # subdomain only on first row
                len(urls) if i == 0 else "",
                round(pct_host, 1) if i == 0 else "",
                pattern,
                count,
                round(count / len(urls) * 100, 1),
                examples[0] if len(examples) > 0 else "",
                examples[1] if len(examples) > 1 else "",
                examples[2] if len(examples) > 2 else "",
            ]
            ws.append(row_data)
            r = ws.max_row

            # Shade subdomain rows
            if i == 0:
                for col in range(1, len(COLUMNS) + 1):
                    ws.cell(row=r, column=col).font = SUBDOMAIN_FONT
                    ws.cell(row=r, column=col).fill = SUBDOMAIN_FILL

        # Blank separator row between subdomains
        ws.append([""] * len(COLUMNS))

wb.save(OUTPUT_FILE)
print(f"Saved to {OUTPUT_FILE}")


Saved to url_analysis.xlsx


transcripts.cnn.com is a section of the CNN website that provides written transcripts of CNN’s TV broadcasts.

What it contains
- Full text transcripts of CNN programs (news shows, interviews, debates, special reports)
- Coverage from shows like Anderson Cooper 360°, The Situation Room, etc.
- Transcripts of breaking news events and live coverage
- Archives organized by date, show, and segment

cnnpressroom.blogs.cnn.com is essentially CNN’s official press and communications site. not a regular news site for the public, but a place where CNN publishes announcements about itself and its content.

Basically, Aljazeera and Politifact have useful results, the times have few, while cnn and france24 only have content that is not directly news articles. cnn have some transcripts of their shows (~2000) which contain useful info.

## Step 3 & 4: Fetch WARC records and extract JSON-LD

For each page found in the index, we:
1. Send an HTTP range request to download just that one WARC record (~50-200KB)
2. Parse the HTML and extract `<script type="application/ld+json">` blocks

In [1]:
import io
import json
import time
import pandas as pd

import duckdb
import requests
from bs4 import BeautifulSoup
from warcio.archiveiterator import ArchiveIterator

CC_DATA_BASE = "https://data.commoncrawl.org"

CRAWL_ID = "CC-MAIN-2026-12"                    # latest crawl as of April 2026
OUTPUT_FILE = "jsonld_results.jsonl"

In [2]:
def fetch_warc_record(filename, offset, length):
    """Fetch a single WARC record via HTTP range request, return HTML or None."""
    url = f"{CC_DATA_BASE}/{filename}"
    headers = {"Range": f"bytes={offset}-{offset + length - 1}"}
    try:
        resp = requests.get(url, headers=headers, timeout=60)
        resp.raise_for_status()
    except requests.RequestException as e:
        print(f"  FETCH ERROR: {type(e).__name__}: {e}") 
        return None

    raw = io.BytesIO(resp.content)
    try:
        for rec in ArchiveIterator(raw):
            if rec.rec_type == "response":
                html_bytes = rec.content_stream().read()
                try:
                    return html_bytes.decode("utf-8")
                except UnicodeDecodeError:
                    return html_bytes.decode("latin-1")
    except Exception:
        return None
    return None


def extract_jsonld(html):
    """Extract all JSON-LD blocks from HTML."""
    soup = BeautifulSoup(html, "html.parser")
    results = []
    for script in soup.find_all("script", attrs={"type": "application/ld+json"}):
        text = script.string
        if not text:
            continue
        try:
            data = json.loads(text)
            if isinstance(data, dict):
                results.append(data)
            elif isinstance(data, list):
                results.extend(data)
        except json.JSONDecodeError:
            pass
    return results

In [ ]:
# import threading
# import time, threading, random
# from concurrent.futures import ThreadPoolExecutor, as_completed

# OUTPUT_FILE = "jsonld_results.jsonl"
# NUM_WORKERS = 16       # concurrent HTTP requests; tune up/down based on rate limits
# SLEEP_PER_REQ = 0.0   # set to e.g. 0.1 if you hit 429s
# MAX_RETRIES = 3

# records_wiki = pd.read_csv("records_wiki.csv")
# records = records_wiki.to_dict("records")   # remove [:20] for full run

# stats = {"fetched": 0, "with_jsonld": 0, "total_blocks": 0, "failed": 0}
# stats_lock = threading.Lock()
# file_lock  = threading.Lock()

# def fetch_warc_record_with_retry(filename, offset, length):
#     for attempt in range(MAX_RETRIES):
#         html = fetch_warc_record(filename, offset, length)
#         if html is not None:
#             return html
#         wait = 2 ** attempt + random.uniform(0, 1)   # 1s, 2s, 4s
#         time.sleep(wait)
#     return None

# def process_record(rec):
#     html = fetch_warc_record_with_retry(
#         rec["warc_filename"], rec["warc_record_offset"], rec["warc_record_length"]
#     )
#     if html is None:
#         return rec["url"], None, None
#     return rec["url"], extract_jsonld(html), rec["url_host_registered_domain"]

# with open(OUTPUT_FILE, "w") as f_out, ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
#     futures = {executor.submit(process_record, rec): i for i, rec in enumerate(records)}

#     for done_count, future in enumerate(as_completed(futures), 1):
#         result = future.result()
#         url = result[0]

#         if result[1] is None:   # fetch failed
#             with stats_lock:
#                 stats["failed"] += 1
#             if done_count % 1000 == 0:
#                 print(f"[{done_count}/{len(records)}] progress | {stats}")
#             continue

#         url, jsonld_blocks, domain = result
#         with stats_lock:
#             stats["fetched"] += 1
#             if jsonld_blocks:
#                 stats["with_jsonld"] += 1
#                 stats["total_blocks"] += len(jsonld_blocks)

#         if jsonld_blocks:
#             lines = [
#                 json.dumps({"url": url, "domain": domain, "jsonld": block}, ensure_ascii=False)
#                 for block in jsonld_blocks
#             ]
#             with file_lock:
#                 f_out.write("\n".join(lines) + "\n")

#         if done_count % 1000 == 0:
#             print(f"[{done_count}/{len(records)}] {stats}")

# print(f"\nDone! {stats}")


  FETCH ERROR: ClientError: An error occurred (AccessDenied) when calling the GetObject operation: Access Denied
  FETCH ERROR: ClientError: An error occurred (AccessDenied) when calling the GetObject operation: Access Denied
  FETCH ERROR: ClientError: An error occurred (AccessDenied) when calling the GetObject operation: Access Denied
  FETCH ERROR: ClientError: An error occurred (AccessDenied) when calling the GetObject operation: Access Denied
  FETCH ERROR: ClientError: An error occurred (AccessDenied) when calling the GetObject operation: Access Denied
  FETCH ERROR: ClientError: An error occurred (AccessDenied) when calling the GetObject operation: Access Denied
  FETCH ERROR: ClientError: An error occurred (AccessDenied) when calling the GetObject operation: Access Denied
  FETCH ERROR: ClientError: An error occurred (AccessDenied) when calling the GetObject operation: Access Denied
  FETCH ERROR: ClientError: An error occurred (AccessDenied) when calling the GetObject operatio

In [ ]:
import io, json, time, threading, random, requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from warcio.archiveiterator import ArchiveIterator
from bs4 import BeautifulSoup

OUTPUT_FILE    = "jsonld_results.jsonl"
NUM_WORKERS    = 8      # 4 workers × 0.5s sleep ≈ 8 req/s — stays under CC's limit
SLEEP_PER_REQ  = 0.25   # sleep after every successful fetch
MAX_RETRIES    = 4

HEADERS = {
    "User-Agent": "research-bot/1.0 (academic thesis; contact: marwaslmn99@gmail.com)"
}

def fetch_warc_record(filename, offset, length):
    url = f"{CC_DATA_BASE}/{filename}"
    headers = {**HEADERS, "Range": f"bytes={offset}-{offset + length - 1}"}
    for attempt in range(MAX_RETRIES):
        try:
            resp = requests.get(url, headers=headers, timeout=60)
            if resp.status_code == 403:
                wait = 60 * (attempt + 1)   # 60s, 120s, 180s — let the block expire
                print(f"  429/403 — waiting {wait}s before retry {attempt+1}")
                time.sleep(wait)
                continue
            resp.raise_for_status()
        except requests.RequestException as e:
            wait = 2 ** attempt + random.uniform(0, 1)
            print(f"  FETCH ERROR attempt {attempt+1}: {type(e).__name__}: {e} — retrying in {wait:.1f}s")
            time.sleep(wait)
            continue

        raw = io.BytesIO(resp.content)
        try:
            for rec in ArchiveIterator(raw):
                if rec.rec_type == "response":
                    html_bytes = rec.content_stream().read()
                    try:
                        return html_bytes.decode("utf-8")
                    except UnicodeDecodeError:
                        return html_bytes.decode("latin-1")
        except Exception:
            return None
        return None
    return None


def extract_jsonld(html):
    soup = BeautifulSoup(html, "html.parser")
    results = []
    for script in soup.find_all("script", attrs={"type": "application/ld+json"}):
        text = script.string
        if not text:
            continue
        try:
            data = json.loads(text)
            if isinstance(data, dict):
                results.append(data)
            elif isinstance(data, list):
                results.extend(data)
        except json.JSONDecodeError:
            pass
    return results


def process_record(rec):
    html = fetch_warc_record(
        rec["warc_filename"], rec["warc_record_offset"], rec["warc_record_length"]
    )
    if SLEEP_PER_REQ:
        time.sleep(SLEEP_PER_REQ)
    if html is None:
        return rec["url"], None, None
    return rec["url"], extract_jsonld(html), rec["url_host_registered_domain"]


# ── main ──────────────────────────────────────────────────────────
import pandas as pd

records_wiki = pd.read_csv("records_wiki.csv")
records = records_wiki.to_dict("records")

stats      = {"fetched": 0, "with_jsonld": 0, "total_blocks": 0, "failed": 0}
stats_lock = threading.Lock()
file_lock  = threading.Lock()

with open(OUTPUT_FILE, "w") as f_out, ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
    futures = {executor.submit(process_record, rec): i for i, rec in enumerate(records)}

    for done_count, future in enumerate(as_completed(futures), 1):
        url, jsonld_blocks, domain = future.result()

        with stats_lock:
            if jsonld_blocks is None:
                stats["failed"] += 1
            else:
                stats["fetched"] += 1
                if jsonld_blocks:
                    stats["with_jsonld"] += 1
                    stats["total_blocks"] += len(jsonld_blocks)

        if jsonld_blocks:
            lines = [
                json.dumps({"url": url, "domain": domain, "jsonld": block}, ensure_ascii=False)
                for block in jsonld_blocks
            ]
            with file_lock:
                f_out.write("\n".join(lines) + "\n")

        if done_count % 500 == 0:
            print(f"[{done_count}/{len(records)}] {stats}")

print(f"\nDone! {stats}")


[500/102422] {'fetched': 500, 'with_jsonld': 482, 'total_blocks': 482, 'failed': 0}
[1000/102422] {'fetched': 1000, 'with_jsonld': 982, 'total_blocks': 982, 'failed': 0}
[1500/102422] {'fetched': 1500, 'with_jsonld': 1482, 'total_blocks': 1482, 'failed': 0}
  429/403 — waiting 60s before retry 1
  429/403 — waiting 60s before retry 1
  429/403 — waiting 60s before retry 1
  429/403 — waiting 60s before retry 1
  429/403 — waiting 60s before retry 1
  429/403 — waiting 60s before retry 1
  429/403 — waiting 60s before retry 1
  429/403 — waiting 60s before retry 1
  429/403 — waiting 120s before retry 2
  429/403 — waiting 120s before retry 2
  429/403 — waiting 120s before retry 2
  429/403 — waiting 120s before retry 2
  429/403 — waiting 120s before retry 2
  429/403 — waiting 120s before retry 2
  429/403 — waiting 120s before retry 2
  429/403 — waiting 120s before retry 2
[2000/102422] {'fetched': 2000, 'with_jsonld': 1982, 'total_blocks': 1982, 'failed': 0}
[2500/102422] {'fetche

## Summary & Quick Analysis

In [ ]:
print(f"Crawl:               {CRAWL_ID}")
print(f"Domain:              {DOMAIN}")
print(f"URL prefix:          {URL_PREFIX or '(all)'}")
print(f"Pages in index:      {len(records)}")
print(f"Pages fetched:       {stats['fetched']}")
print(f"Pages with JSON-LD:  {stats['with_jsonld']}")
print(f"Total JSON-LD blocks:{stats['total_blocks']}")
print(f"Output file:         {OUTPUT_FILE}")

if stats["fetched"] > 0:
    pct = stats["with_jsonld"] / stats["fetched"] * 100
    print(f"\n% of pages with JSON-LD: {pct:.1f}%")

In [ ]:
from collections import Counter

type_counter = Counter()
for entry in all_results:
    t = entry["jsonld"].get("@type", "unknown")
    if isinstance(t, list):
        for x in t:
            type_counter[x] += 1
    else:
        type_counter[t] += 1

print("@type distribution:")
for typ, count in type_counter.most_common(20):
    print(f"  {typ}: {count}")

In [ ]:
prop_counter = Counter()
for entry in all_results:
    for key in entry["jsonld"].keys():
        if not key.startswith("@"):
            prop_counter[key] += 1

print("Top 30 properties used:")
for prop, count in prop_counter.most_common(30):
    print(f"  {prop}: {count}")

In [ ]:
if all_results:
    print(json.dumps(all_results[0]["jsonld"], indent=2, ensure_ascii=False)[:3000])